# Priprema podataka

Cilj ovog notebook-a je formiranje analitickih tabela koje ce se koristiti u nastavku projekta.

Originalni Olist skup podataka je relacionog oblika, pa se informacije o kupcima, prodavcima, porudzbinama, placanjima, proizvodima i ocenama nalaze u vise povezanih tabela. Zbog toga je pre primene algoritama potrebno objediniti podatke i izvrsiti agregacije na odgovarajucem nivou.

U ovom notebook-u formiraju se tri glavne tabele:

- `customer_master` - tabela za klasterovanje kupaca
- `seller_master` - tabela za klasterovanje prodavaca
- `satisfaction_table` - tabela za klasifikaciju zadovoljstva kupaca

Ove tabele predstavljaju osnovu za naredne notebook-e u kojima ce biti primenjeni algoritmi klasterovanja i klasifikacije.

## Ucitavanje biblioteka i podataka

U prvom koraku ucitavaju se potrebne biblioteke i originalne tabele iz Olist skupa podataka.

In [1]:
import os
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

In [2]:
RAW_DATA_PATH = "../data/raw"
PROCESSED_DATA_PATH = "../data/processed"

os.makedirs(PROCESSED_DATA_PATH, exist_ok=True)

In [3]:
customers = pd.read_csv(f"{RAW_DATA_PATH}/olist_customers_dataset.csv")
orders = pd.read_csv(f"{RAW_DATA_PATH}/olist_orders_dataset.csv")
items = pd.read_csv(f"{RAW_DATA_PATH}/olist_order_items_dataset.csv")
payments = pd.read_csv(f"{RAW_DATA_PATH}/olist_order_payments_dataset.csv")
reviews = pd.read_csv(f"{RAW_DATA_PATH}/olist_order_reviews_dataset.csv")
products = pd.read_csv(f"{RAW_DATA_PATH}/olist_products_dataset.csv")
sellers = pd.read_csv(f"{RAW_DATA_PATH}/olist_sellers_dataset.csv")
translation = pd.read_csv(f"{RAW_DATA_PATH}/product_category_name_translation.csv")

In [4]:
datasets = {
    "customers": customers,
    "orders": orders,
    "items": items,
    "payments": payments,
    "reviews": reviews,
    "products": products,
    "sellers": sellers,
    "translation": translation
}

pd.DataFrame({
    "table": list(datasets.keys()),
    "rows": [df.shape[0] for df in datasets.values()],
    "columns": [df.shape[1] for df in datasets.values()]
})

,table,rows,columns
0,customers,99441,5
1,orders,99441,8
2,items,112650,7
3,payments,103886,5
4,reviews,99224,7
5,products,32951,9
6,sellers,3095,4
7,translation,71,2


## Pomocne funkcije

U nastavku su definisane pomocne funkcije koje se koriste vise puta u notebook-u. Funkcija `mode_or_unknown` vraca najcescu vrednost u koloni, dok funkcija `fill_missing_values` popunjava nedostajuce vrednosti medijanom za numericke atribute i vrednoscu `unknown` za kategoricke atribute.

In [5]:
def mode_or_unknown(series):
    series = series.dropna()
    if series.empty:
        return "unknown"
    return series.mode().iloc[0]


def fill_missing_values(df):
    df = df.copy()

    numeric_columns = df.select_dtypes(include=["number"]).columns
    categorical_columns = df.select_dtypes(include=["object", "category"]).columns

    for column in numeric_columns:
        df[column] = df[column].fillna(df[column].median())

    for column in categorical_columns:
        df[column] = df[column].fillna("unknown")

    return df

## Obrada vremenskih atributa

Vremenski atributi koriste se za racunanje trajanja isporuke, kasnjenja, aktivnosti kupaca i aktivnosti prodavaca. Zbog toga je neophodno pretvoriti ih u `datetime` format.

In [6]:
order_date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for column in order_date_columns:
    orders[column] = pd.to_datetime(orders[column], errors="coerce")

items["shipping_limit_date"] = pd.to_datetime(items["shipping_limit_date"], errors="coerce")

## Priprema kategorija proizvoda

Originalni nazivi kategorija proizvoda su na portugalskom jeziku. Zbog lakse interpretacije rezultata koriste se engleski prevodi iz tabele `product_category_name_translation`.

In [7]:
products_en = products.merge(
    translation,
    on="product_category_name",
    how="left"
)

products_en["product_category_name_english"] = (
    products_en["product_category_name_english"]
    .fillna(products_en["product_category_name"])
    .fillna("unknown")
)

products_en = products_en[[
    "product_id",
    "product_category_name_english"
]]

products_en.head()

,product_id,product_category_name_english
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,art
2,96bd76ec8810374ed1b65e291975717f,sports_leisure
3,cef67bcfe19066a932b7673e239eb23d,baby
4,9dc1a7de274444849c219cff195d0b71,housewares


## Formiranje tabele na nivou porudzbine

Pre formiranja master tabela potrebno je objediniti informacije na nivou porudzbine. Jedna porudzbina moze sadrzati vise proizvoda i vise zapisa o placanju, pa se ove informacije najpre agregiraju po `order_id`.

In [8]:
items_with_categories = items.merge(
    products_en,
    on="product_id",
    how="left"
)

items_with_categories["product_category_name_english"] = (
    items_with_categories["product_category_name_english"].fillna("unknown")
)

In [9]:
items_agg = (
    items_with_categories
    .groupby("order_id")
    .agg(
        items_count=("order_item_id", "count"),
        total_items_value=("price", "sum"),
        total_freight_value=("freight_value", "sum"),
        avg_item_price=("price", "mean"),
        unique_products=("product_id", "nunique"),
        unique_sellers=("seller_id", "nunique"),
        main_category=("product_category_name_english", mode_or_unknown),
        unique_categories=("product_category_name_english", "nunique")
    )
    .reset_index()
)

items_agg.head()

,order_id,items_count,total_items_value,total_freight_value,avg_item_price,unique_products,unique_sellers,main_category,unique_categories
0,00010242fe8c5a6d1ba2dd792cb16214,1,58.90,13.29,58.90,1,1,cool_stuff,1
1,00018f77f2f0320c557190d7a144bdd3,1,239.90,19.93,239.90,1,1,pet_shop,1
2,000229ec398224ef6ca0657da4fc703e,1,199.00,17.87,199.00,1,1,furniture_decor,1
3,00024acbcdf0a6daa1e931b038114c75,1,12.99,12.79,12.99,1,1,perfumery,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,199.90,18.14,199.90,1,1,garden_tools,1


In [10]:
payments_agg = (
    payments
    .groupby("order_id")
    .agg(
        total_payment_value=("payment_value", "sum"),
        avg_payment_installments=("payment_installments", "mean"),
        max_payment_installments=("payment_installments", "max"),
        preferred_payment_type=("payment_type", mode_or_unknown)
    )
    .reset_index()
)

payments_agg.head()

,order_id,total_payment_value,avg_payment_installments,max_payment_installments,preferred_payment_type
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,2.0,2,credit_card
1,00018f77f2f0320c557190d7a144bdd3,259.83,3.0,3,credit_card
2,000229ec398224ef6ca0657da4fc703e,216.87,5.0,5,credit_card
3,00024acbcdf0a6daa1e931b038114c75,25.78,2.0,2,credit_card
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,3.0,3,credit_card


In [11]:
reviews_agg = (
    reviews
    .groupby("order_id")
    .agg(
        review_score=("review_score", "mean")
    )
    .reset_index()
)

reviews_agg.head()

,order_id,review_score
0,00010242fe8c5a6d1ba2dd792cb16214,5.0
1,00018f77f2f0320c557190d7a144bdd3,4.0
2,000229ec398224ef6ca0657da4fc703e,5.0
3,00024acbcdf0a6daa1e931b038114c75,4.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,5.0


In [12]:
order_table = (
    orders
    .merge(customers, on="customer_id", how="left")
    .merge(items_agg, on="order_id", how="left")
    .merge(payments_agg, on="order_id", how="left")
    .merge(reviews_agg, on="order_id", how="left")
)

order_table["delivery_days"] = (
    order_table["order_delivered_customer_date"] -
    order_table["order_purchase_timestamp"]
).dt.days

order_table["estimated_delivery_days"] = (
    order_table["order_estimated_delivery_date"] -
    order_table["order_purchase_timestamp"]
).dt.days

order_table["delay_days"] = (
    order_table["order_delivered_customer_date"] -
    order_table["order_estimated_delivery_date"]
).dt.days

order_table["is_delayed"] = (order_table["delay_days"] > 0).astype(int)
order_table["order_month"] = order_table["order_purchase_timestamp"].dt.month
order_table["order_year"] = order_table["order_purchase_timestamp"].dt.year

order_table.shape

(99441, 31)

Tabela `order_table` predstavlja pomocnu tabelu na nivou porudzbine. Ona se ne koristi kao krajnji skup podataka, vec kao osnova za formiranje tabela `customer_master`, `seller_master` i `satisfaction_table`.

In [13]:
order_table.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,items_count,total_items_value,total_freight_value,avg_item_price,unique_products,unique_sellers,main_category,unique_categories,total_payment_value,avg_payment_installments,max_payment_installments,preferred_payment_type,review_score,delivery_days,estimated_delivery_days,delay_days,is_delayed,order_month,order_year
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,1.0,29.99,8.72,29.99,1.0,1.0,housewares,1.0,38.71,1.0,1.0,voucher,4.0,8.0,15,-8.0,0,10,2017
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,1.0,118.70,22.76,118.70,1.0,1.0,perfumery,1.0,141.46,1.0,1.0,boleto,4.0,13.0,19,-6.0,0,7,2018
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,1.0,159.90,19.22,159.90,1.0,1.0,auto,1.0,179.12,3.0,3.0,credit_card,5.0,9.0,26,-18.0,0,8,2018
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,1.0,45.00,27.20,45.00,1.0,1.0,pet_shop,1.0,72.20,1.0,1.0,credit_card,5.0,13.0,26,-13.0,0,11,2017
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,1.0,19.90,8.72,19.90,1.0,1.0,stationery,1.0,28.62,1.0,1.0,credit_card,5.0,2.0,12,-10.0,0,2,2018


# Customer master tabela

Tabela `customer_master` sadrzi jedan red za svakog jedinstvenog kupca, odnosno za svaku vrednost atributa `customer_unique_id`.

Ova tabela se koristi za klasterovanje kupaca. Atributi opisuju aktivnost kupca, potrosnju, placanje, ocene, isporuku i osnovne preferencije.

In [14]:
customer_master = (
    order_table
    .groupby("customer_unique_id")
    .agg(
        order_count=("order_id", "nunique"),
        total_spent=("total_payment_value", "sum"),
        total_items_value=("total_items_value", "sum"),
        total_freight_value=("total_freight_value", "sum"),
        avg_order_value=("total_payment_value", "mean"),
        avg_item_price=("avg_item_price", "mean"),
        avg_freight_value=("total_freight_value", "mean"),
        avg_items_count=("items_count", "mean"),
        avg_payment_installments=("avg_payment_installments", "mean"),
        avg_review_score=("review_score", "mean"),
        avg_delivery_days=("delivery_days", "mean"),
        avg_estimated_delivery_days=("estimated_delivery_days", "mean"),
        avg_delay_days=("delay_days", "mean"),
        delayed_order_rate=("is_delayed", "mean"),
        unique_products=("unique_products", "sum"),
        unique_categories=("main_category", "nunique"),
        first_order_date=("order_purchase_timestamp", "min"),
        last_order_date=("order_purchase_timestamp", "max"),
        customer_state=("customer_state", mode_or_unknown),
        preferred_category=("main_category", mode_or_unknown),
        preferred_payment_type=("preferred_payment_type", mode_or_unknown)
    )
    .reset_index()
)

reference_date = order_table["order_purchase_timestamp"].max()

customer_master["recency_days"] = (
    reference_date - customer_master["last_order_date"]
).dt.days

customer_master["customer_lifetime_days"] = (
    customer_master["last_order_date"] - customer_master["first_order_date"]
).dt.days

customer_master = customer_master.drop(columns=["first_order_date", "last_order_date"])

customer_master.head()

,customer_unique_id,order_count,total_spent,total_items_value,total_freight_value,avg_order_value,avg_item_price,avg_freight_value,avg_items_count,avg_payment_installments,avg_review_score,avg_delivery_days,avg_estimated_delivery_days,avg_delay_days,delayed_order_rate,unique_products,unique_categories,customer_state,preferred_category,preferred_payment_type,recency_days,customer_lifetime_days
0,0000366f3b9a7992bf8c76cfdf3221e2,1,141.90,129.90,12.00,141.90,129.90,12.00,1.0,8.0,5.0,6.0,10.0,-5.0,0.0,1.0,1,SP,bed_bath_table,credit_card,160,0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,27.19,18.90,8.29,27.19,18.90,8.29,1.0,1.0,4.0,3.0,7.0,-5.0,0.0,1.0,1,SP,health_beauty,credit_card,163,0
2,0000f46a3911fa3c0805444483337064,1,86.22,69.00,17.22,86.22,69.00,17.22,1.0,8.0,3.0,25.0,27.0,-2.0,0.0,1.0,1,SC,stationery,credit_card,585,0
3,0000f6ccb0745a6a4b88665a16c9f078,1,43.62,25.99,17.63,43.62,25.99,17.63,1.0,4.0,4.0,20.0,31.0,-12.0,0.0,1.0,1,PA,telephony,credit_card,369,0
4,0004aac84e0df4da2b147fca70cf8255,1,196.89,180.00,16.89,196.89,180.00,16.89,1.0,6.0,5.0,13.0,20.0,-8.0,0.0,1.0,1,SP,telephony,credit_card,336,0


## Provera customer_master tabele

Nakon formiranja tabele proveravaju se dimenzije, osnovne statistike i nedostajuce vrednosti. Nedostajuce vrednosti nastaju uglavnom kod porudzbina kod kojih ne postoje sve informacije o isporuci ili ocenama.

In [15]:
customer_master.shape

(96096, 22)

In [16]:
customer_master.describe().T

,count,mean,std,min,25%,50%,75%,max
order_count,96096.0,1.034809,0.214384,1.00,1.00,1.000,1.00,17.00
total_spent,96096.0,166.592492,231.428332,0.00,63.12,108.000,183.53,13664.08
total_items_value,96096.0,141.438184,217.215904,0.00,45.99,89.000,154.00,13440.00
total_freight_value,96096.0,23.433957,22.883215,0.00,13.92,17.530,25.42,1794.96
avg_order_value,96095.0,161.401796,222.308078,0.00,62.46,105.830,177.21,13664.08
avg_item_price,95420.0,126.555747,191.751327,0.85,42.90,79.445,139.90,6735.00
avg_freight_value,95420.0,22.819050,21.560878,0.00,13.89,17.240,24.13,1794.96
avg_items_count,95420.0,1.139100,0.526880,1.00,1.00,1.000,1.00,21.00
avg_payment_installments,96095.0,2.901885,2.677704,0.00,1.00,2.000,4.00,24.00
avg_review_score,95380.0,4.084963,1.341661,1.00,4.00,5.000,5.00,5.00


In [17]:
customer_missing = (
    customer_master
    .isnull()
    .sum()
    .reset_index()
)
customer_missing.columns = ["column", "missing_count"]
customer_missing["missing_percent"] = round(customer_missing["missing_count"] / len(customer_master) * 100, 2)
customer_missing = customer_missing[customer_missing["missing_count"] > 0]
customer_missing.sort_values("missing_percent", ascending=False)

,column,missing_count,missing_percent
11,avg_delivery_days,2740,2.85
13,avg_delay_days,2740,2.85
10,avg_review_score,716,0.75
6,avg_item_price,676,0.70
7,avg_freight_value,676,0.70
8,avg_items_count,676,0.70
5,avg_order_value,1,0.00
9,avg_payment_installments,1,0.00


Nedostajuce vrednosti se popunjavaju pre cuvanja tabele. Numericki atributi se popunjavaju medijanom, dok se kategoricki atributi popunjavaju vrednoscu `unknown`. Skaliranje se ne vrsi u ovom notebook-u, vec u notebook-u za klasterovanje.

In [18]:
customer_master = fill_missing_values(customer_master)

customer_master.to_csv(
    f"{PROCESSED_DATA_PATH}/customer_master.csv",
    index=False
)

customer_master.head()

,customer_unique_id,order_count,total_spent,total_items_value,total_freight_value,avg_order_value,avg_item_price,avg_freight_value,avg_items_count,avg_payment_installments,avg_review_score,avg_delivery_days,avg_estimated_delivery_days,avg_delay_days,delayed_order_rate,unique_products,unique_categories,customer_state,preferred_category,preferred_payment_type,recency_days,customer_lifetime_days
0,0000366f3b9a7992bf8c76cfdf3221e2,1,141.90,129.90,12.00,141.90,129.90,12.00,1.0,8.0,5.0,6.0,10.0,-5.0,0.0,1.0,1,SP,bed_bath_table,credit_card,160,0
1,0000b849f77a49e4a4ce2b2a4ca5be3f,1,27.19,18.90,8.29,27.19,18.90,8.29,1.0,1.0,4.0,3.0,7.0,-5.0,0.0,1.0,1,SP,health_beauty,credit_card,163,0
2,0000f46a3911fa3c0805444483337064,1,86.22,69.00,17.22,86.22,69.00,17.22,1.0,8.0,3.0,25.0,27.0,-2.0,0.0,1.0,1,SC,stationery,credit_card,585,0
3,0000f6ccb0745a6a4b88665a16c9f078,1,43.62,25.99,17.63,43.62,25.99,17.63,1.0,4.0,4.0,20.0,31.0,-12.0,0.0,1.0,1,PA,telephony,credit_card,369,0
4,0004aac84e0df4da2b147fca70cf8255,1,196.89,180.00,16.89,196.89,180.00,16.89,1.0,6.0,5.0,13.0,20.0,-8.0,0.0,1.0,1,SP,telephony,credit_card,336,0


# Seller master tabela

Tabela `seller_master` sadrzi jedan red za svakog prodavca, odnosno za svaku vrednost atributa `seller_id`.

Ova tabela se koristi za klasterovanje prodavaca. Atributi opisuju obim prodaje, prihod, isporuku, ocene kupaca i raznovrsnost proizvoda.

In [19]:
seller_orders = (
    items_with_categories
    .merge(orders[[
        "order_id",
        "customer_id",
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "order_status"
    ]], on="order_id", how="left")
    .merge(customers[[
        "customer_id",
        "customer_unique_id",
        "customer_state"
    ]], on="customer_id", how="left")
    .merge(sellers[[
        "seller_id",
        "seller_state"
    ]], on="seller_id", how="left")
    .merge(reviews_agg, on="order_id", how="left")
)

seller_orders["delivery_days"] = (
    seller_orders["order_delivered_customer_date"] -
    seller_orders["order_purchase_timestamp"]
).dt.days

seller_orders["delay_days"] = (
    seller_orders["order_delivered_customer_date"] -
    seller_orders["order_estimated_delivery_date"]
).dt.days

seller_orders["is_delayed"] = (seller_orders["delay_days"] > 0).astype(int)

seller_orders.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name_english,customer_id,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,order_status,customer_unique_id,customer_state,seller_state,review_score,delivery_days,delay_days,is_delayed
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,cool_stuff,3ce436f183e68e07877b285a838db11a,2017-09-13 08:59:02,2017-09-20 23:43:48,2017-09-29,delivered,871766c5855e863f6eccc05f988b23cb,RJ,SP,5.0,7.0,-9.0,0
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,pet_shop,f6dd3ec061db4e3987629fe6b26e5cce,2017-04-26 10:53:06,2017-05-12 16:04:24,2017-05-15,delivered,eb28e67c4c0b83846050ddfb8a35d051,SP,SP,4.0,16.0,-3.0,0
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,furniture_decor,6489ae5e4333f3693df5ad4372dab6d3,2018-01-14 14:33:31,2018-01-22 13:19:16,2018-02-05,delivered,3818d81c6709e39d06b2738a8d3a2474,MG,MG,5.0,7.0,-14.0,0
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,perfumery,d4eb9395c8c0431ee92fce09860c5a06,2018-08-08 10:00:35,2018-08-14 13:32:39,2018-08-20,delivered,af861d436cfc08b2c2ddefd0ba074622,SP,SP,4.0,6.0,-6.0,0
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,garden_tools,58dbd0b2d70206bf40e62cd34e84d795,2017-02-04 13:57:51,2017-03-01 16:42:31,2017-03-17,delivered,64b576fb70d441e8f1b2d7d446e483c5,SP,PR,5.0,25.0,-16.0,0


In [20]:
seller_master = (
    seller_orders
    .groupby("seller_id")
    .agg(
        seller_order_count=("order_id", "nunique"),
        seller_items_sold=("order_item_id", "count"),
        seller_unique_customers=("customer_unique_id", "nunique"),
        seller_total_revenue=("price", "sum"),
        seller_total_freight_value=("freight_value", "sum"),
        seller_avg_item_price=("price", "mean"),
        seller_avg_freight_value=("freight_value", "mean"),
        seller_avg_review_score=("review_score", "mean"),
        seller_avg_delivery_days=("delivery_days", "mean"),
        seller_avg_delay_days=("delay_days", "mean"),
        seller_delayed_order_rate=("is_delayed", "mean"),
        seller_unique_products=("product_id", "nunique"),
        seller_unique_categories=("product_category_name_english", "nunique"),
        first_sale_date=("order_purchase_timestamp", "min"),
        last_sale_date=("order_purchase_timestamp", "max"),
        seller_state=("seller_state", mode_or_unknown),
        seller_top_category=("product_category_name_english", mode_or_unknown)
    )
    .reset_index()
)

seller_master["seller_active_days"] = (
    seller_master["last_sale_date"] - seller_master["first_sale_date"]
).dt.days

seller_master = seller_master.drop(columns=["first_sale_date", "last_sale_date"])

seller_master.head()

,seller_id,seller_order_count,seller_items_sold,seller_unique_customers,seller_total_revenue,seller_total_freight_value,seller_avg_item_price,seller_avg_freight_value,seller_avg_review_score,seller_avg_delivery_days,seller_avg_delay_days,seller_delayed_order_rate,seller_unique_products,seller_unique_categories,seller_state,seller_top_category,seller_active_days
0,0015a82c2db000af6aaaf3ae2ecb0532,3,3,3,2685.00,63.06,895.000000,21.020000,3.666667,10.333333,-16.333333,0.000000,1,1,SP,small_appliances,21
1,001cca7ae9ae17fb1caed9dfb1094831,200,239,200,25080.03,8854.14,104.937364,37.046611,3.902542,12.628205,-13.213675,0.050209,11,2,ES,garden_tools,523
2,001e6ad469a905060d959994f1b41e4f,1,1,1,250.00,17.94,250.000000,17.940000,1.000000,NaN,NaN,0.000000,1,1,RJ,sports_leisure,0
3,002100f778ceb8431b7a1020ff7ab48f,51,55,51,1234.50,793.66,22.445455,14.430182,3.981818,15.777778,-8.185185,0.163636,24,1,SP,furniture_decor,210
4,003554e2dce176b5555353e4f3555ac8,1,1,1,120.00,19.38,120.000000,19.380000,5.000000,4.000000,-27.000000,0.000000,1,1,GO,unknown,0


## Provera seller_master tabele

Tabela prodavaca je znatno manja od tabele kupaca, sto je vazno za izbor algoritama u nastavku. Zbog toga ce se za kupce koristiti algoritmi pogodniji za veci broj instanci, dok ce se za prodavce koristiti i hijerarhijski pristup.

In [21]:
seller_master.shape

(3095, 17)

In [22]:
seller_master.describe().T

,count,mean,std,min,25%,50%,75%,max
seller_order_count,3095.0,32.313409,105.139763,1.00,2.000000,6.000000,21.500000,1854.000000
seller_items_sold,3095.0,36.397415,119.193461,1.00,2.000000,8.000000,24.000000,2033.000000
seller_unique_customers,3095.0,32.001939,103.766806,1.00,2.000000,6.000000,21.000000,1824.000000
seller_total_revenue,3095.0,4391.484233,13921.997192,3.50,208.850000,821.480000,3280.830000,229472.630000
seller_total_freight_value,3095.0,727.595974,2327.157672,6.66,46.170000,155.520000,558.170000,51612.550000
seller_avg_item_price,3095.0,176.325102,322.145495,3.50,52.183608,95.465385,173.992857,6729.000000
seller_avg_freight_value,3095.0,23.380116,18.957766,1.20,14.740000,18.230000,24.368333,308.336667
seller_avg_review_score,3090.0,3.972987,0.971256,1.00,3.714286,4.166667,4.600000,5.000000
seller_avg_delivery_days,2970.0,11.705458,7.111951,1.00,7.944504,10.691851,13.828125,189.000000
seller_avg_delay_days,2970.0,-12.287716,7.998936,-66.00,-15.433338,-12.000000,-9.000000,167.000000


In [23]:
seller_missing = (
    seller_master
    .isnull()
    .sum()
    .reset_index()
)
seller_missing.columns = ["column", "missing_count"]
seller_missing["missing_percent"] = round(seller_missing["missing_count"] / len(seller_master) * 100, 2)
seller_missing = seller_missing[seller_missing["missing_count"] > 0]
seller_missing.sort_values("missing_percent", ascending=False)

,column,missing_count,missing_percent
9,seller_avg_delivery_days,125,4.04
10,seller_avg_delay_days,125,4.04
8,seller_avg_review_score,5,0.16


Kao i kod kupaca, nedostajuce vrednosti se popunjavaju pre cuvanja tabele. Skaliranje i eventualna transformacija atributa bice uradjeni u notebook-u za klasterovanje prodavaca.

In [24]:
seller_master = fill_missing_values(seller_master)

seller_master.to_csv(
    f"{PROCESSED_DATA_PATH}/seller_master.csv",
    index=False
)

seller_master.head()

,seller_id,seller_order_count,seller_items_sold,seller_unique_customers,seller_total_revenue,seller_total_freight_value,seller_avg_item_price,seller_avg_freight_value,seller_avg_review_score,seller_avg_delivery_days,seller_avg_delay_days,seller_delayed_order_rate,seller_unique_products,seller_unique_categories,seller_state,seller_top_category,seller_active_days
0,0015a82c2db000af6aaaf3ae2ecb0532,3,3,3,2685.00,63.06,895.000000,21.020000,3.666667,10.333333,-16.333333,0.000000,1,1,SP,small_appliances,21
1,001cca7ae9ae17fb1caed9dfb1094831,200,239,200,25080.03,8854.14,104.937364,37.046611,3.902542,12.628205,-13.213675,0.050209,11,2,ES,garden_tools,523
2,001e6ad469a905060d959994f1b41e4f,1,1,1,250.00,17.94,250.000000,17.940000,1.000000,10.691851,-12.000000,0.000000,1,1,RJ,sports_leisure,0
3,002100f778ceb8431b7a1020ff7ab48f,51,55,51,1234.50,793.66,22.445455,14.430182,3.981818,15.777778,-8.185185,0.163636,24,1,SP,furniture_decor,210
4,003554e2dce176b5555353e4f3555ac8,1,1,1,120.00,19.38,120.000000,19.380000,5.000000,4.000000,-27.000000,0.000000,1,1,GO,unknown,0


# Satisfaction tabela

Tabela `satisfaction_table` formira se na nivou porudzbine i koristi se za klasifikaciju zadovoljstva kupaca.

Ciljna promenljiva se dobija na osnovu ocene korisnika:

- `0` - nezadovoljan kupac, ocene 1 i 2
- `1` - neutralan kupac, ocena 3
- `2` - zadovoljan kupac, ocene 4 i 5

Ova podela zadrzava vise informacija nego binarna klasifikacija i omogucava modelu da razlikuje negativno, neutralno i pozitivno iskustvo kupca.

In [25]:
def satisfaction_class(score):
    if score <= 2:
        return 0
    elif score == 3:
        return 1
    else:
        return 2

In [26]:
satisfaction_table = order_table.copy()

satisfaction_table = satisfaction_table[
    satisfaction_table["review_score"].notnull()
].copy()

satisfaction_table["satisfaction_class"] = (
    satisfaction_table["review_score"]
    .apply(satisfaction_class)
)

satisfaction_columns = [
    "order_id",
    "customer_unique_id",
    "customer_state",
    "order_status",
    "items_count",
    "total_items_value",
    "total_freight_value",
    "avg_item_price",
    "unique_products",
    "unique_sellers",
    "unique_categories",
    "total_payment_value",
    "avg_payment_installments",
    "max_payment_installments",
    "preferred_payment_type",
    "main_category",
    "delivery_days",
    "estimated_delivery_days",
    "delay_days",
    "is_delayed",
    "order_month",
    "order_year",
    "review_score",
    "satisfaction_class"
]

satisfaction_table = satisfaction_table[satisfaction_columns]

satisfaction_table.head()

,order_id,customer_unique_id,customer_state,order_status,items_count,total_items_value,total_freight_value,avg_item_price,unique_products,unique_sellers,unique_categories,total_payment_value,avg_payment_installments,max_payment_installments,preferred_payment_type,main_category,delivery_days,estimated_delivery_days,delay_days,is_delayed,order_month,order_year,review_score,satisfaction_class
0,e481f51cbdc54678b7cc49136f2d6af7,7c396fd4830fd04220f754e42b4e5bff,SP,delivered,1.0,29.99,8.72,29.99,1.0,1.0,1.0,38.71,1.0,1.0,voucher,housewares,8.0,15,-8.0,0,10,2017,4.0,2
1,53cdb2fc8bc7dce0b6741e2150273451,af07308b275d755c9edb36a90c618231,BA,delivered,1.0,118.70,22.76,118.70,1.0,1.0,1.0,141.46,1.0,1.0,boleto,perfumery,13.0,19,-6.0,0,7,2018,4.0,2
2,47770eb9100c2d0c44946d9cf07ec65d,3a653a41f6f9fc3d2a113cf8398680e8,GO,delivered,1.0,159.90,19.22,159.90,1.0,1.0,1.0,179.12,3.0,3.0,credit_card,auto,9.0,26,-18.0,0,8,2018,5.0,2
3,949d5b44dbf5de918fe9c16f97b45f8a,7c142cf63193a1473d2e66489a9ae977,RN,delivered,1.0,45.00,27.20,45.00,1.0,1.0,1.0,72.20,1.0,1.0,credit_card,pet_shop,13.0,26,-13.0,0,11,2017,5.0,2
4,ad21c59c0840e6cb83a9ceb5573f8159,72632f0f9dd73dfee390c9b22eb56dd6,SP,delivered,1.0,19.90,8.72,19.90,1.0,1.0,1.0,28.62,1.0,1.0,credit_card,stationery,2.0,12,-10.0,0,2,2018,5.0,2


## Provera raspodele ciljne promenljive

Pre treniranja modela potrebno je proveriti raspodelu klasa. Kod ocena korisnika ocekivano je da postoji neuravnotezenost, jer vecina kupaca daje visoke ocene.

In [27]:
satisfaction_distribution = (
    satisfaction_table["satisfaction_class"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
    .reset_index()
)

satisfaction_distribution.columns = ["satisfaction_class", "percentage"]
satisfaction_distribution

,satisfaction_class,percentage
0,0,14.64
1,1,8.25
2,2,77.11


Iako je klasa zadovoljnih kupaca dominantna, raspodela nije toliko ekstremna kao kod drugih mogucih ciljnih promenljivih, poput ponovne kupovine, statusa porudzbine ili kasnjenja isporuke. Zbog toga je zadovoljstvo kupaca izabrano kao glavni klasifikacioni problem.

Kolona `review_score` ostaje u tabeli radi provere i interpretacije, ali se ne sme koristiti kao ulazni atribut prilikom treniranja modela jer iz nje direktno nastaje ciljna promenljiva.

In [28]:
satisfaction_missing = (
    satisfaction_table
    .isnull()
    .sum()
    .reset_index()
)
satisfaction_missing.columns = ["column", "missing_count"]
satisfaction_missing["missing_percent"] = round(satisfaction_missing["missing_count"] / len(satisfaction_table) * 100, 2)
satisfaction_missing = satisfaction_missing[satisfaction_missing["missing_count"] > 0]
satisfaction_missing.sort_values("missing_percent", ascending=False)

,column,missing_count,missing_percent
18,delay_days,2843,2.88
16,delivery_days,2843,2.88
6,total_freight_value,756,0.77
7,avg_item_price,756,0.77
4,items_count,756,0.77
5,total_items_value,756,0.77
9,unique_sellers,756,0.77
8,unique_products,756,0.77
10,unique_categories,756,0.77
15,main_category,756,0.77


In [29]:
satisfaction_table = fill_missing_values(satisfaction_table)

satisfaction_table.to_csv(
    f"{PROCESSED_DATA_PATH}/satisfaction_table.csv",
    index=False
)

satisfaction_table.head()

,order_id,customer_unique_id,customer_state,order_status,items_count,total_items_value,total_freight_value,avg_item_price,unique_products,unique_sellers,unique_categories,total_payment_value,avg_payment_installments,max_payment_installments,preferred_payment_type,main_category,delivery_days,estimated_delivery_days,delay_days,is_delayed,order_month,order_year,review_score,satisfaction_class
0,e481f51cbdc54678b7cc49136f2d6af7,7c396fd4830fd04220f754e42b4e5bff,SP,delivered,1.0,29.99,8.72,29.99,1.0,1.0,1.0,38.71,1.0,1.0,voucher,housewares,8.0,15,-8.0,0,10,2017,4.0,2
1,53cdb2fc8bc7dce0b6741e2150273451,af07308b275d755c9edb36a90c618231,BA,delivered,1.0,118.70,22.76,118.70,1.0,1.0,1.0,141.46,1.0,1.0,boleto,perfumery,13.0,19,-6.0,0,7,2018,4.0,2
2,47770eb9100c2d0c44946d9cf07ec65d,3a653a41f6f9fc3d2a113cf8398680e8,GO,delivered,1.0,159.90,19.22,159.90,1.0,1.0,1.0,179.12,3.0,3.0,credit_card,auto,9.0,26,-18.0,0,8,2018,5.0,2
3,949d5b44dbf5de918fe9c16f97b45f8a,7c142cf63193a1473d2e66489a9ae977,RN,delivered,1.0,45.00,27.20,45.00,1.0,1.0,1.0,72.20,1.0,1.0,credit_card,pet_shop,13.0,26,-13.0,0,11,2017,5.0,2
4,ad21c59c0840e6cb83a9ceb5573f8159,72632f0f9dd73dfee390c9b22eb56dd6,SP,delivered,1.0,19.90,8.72,19.90,1.0,1.0,1.0,28.62,1.0,1.0,credit_card,stationery,2.0,12,-10.0,0,2,2018,5.0,2


# Pregled sacuvanih tabela

Na kraju notebook-a proverava se koje su tabele formirane i sacuvane za nastavak projekta.

In [31]:
pd.set_option("display.max_colwidth", None)
prepared_tables = pd.DataFrame({
    "table": [
        "customer_master",
        "seller_master",
        "satisfaction_table"
    ],
    "rows": [
        customer_master.shape[0],
        seller_master.shape[0],
        satisfaction_table.shape[0]
    ],
    "columns": [
        customer_master.shape[1],
        seller_master.shape[1],
        satisfaction_table.shape[1]
    ],
    "purpose": [
        "klasterovanje kupaca - KMeans i BIRCH",
        "klasterovanje prodavaca - Agglomerative i GMM",
        "klasifikacija zadovoljstva kupaca - RandomForest i XGBoost"
    ]
})

prepared_tables

,table,rows,columns,purpose
0,customer_master,96096,22,klasterovanje kupaca - KMeans i BIRCH
1,seller_master,3095,17,klasterovanje prodavaca - Agglomerative i GMM
2,satisfaction_table,98673,24,klasifikacija zadovoljstva kupaca - RandomForest i XGBoost


# Zakljucak

U ovom notebook-u formirane su tri analiticke tabele koje ce biti koriscene u nastavku projekta.

Tabela `customer_master` namenjena je klasterovanju kupaca i sadrzi agregirane informacije o aktivnosti, potrosnji, placanju, isporuci i preferencijama kupaca.

Tabela `seller_master` namenjena je klasterovanju prodavaca i opisuje obim prodaje, prihod, ocene, kasnjenja i raznovrsnost proizvoda.

Tabela `satisfaction_table` koristi se za klasifikaciju zadovoljstva kupaca na osnovu karakteristika porudzbine, placanja i isporuke.

Na ovaj nacin originalni relacioni skup podataka preveden je u oblik pogodan za primenu algoritama istrazivanja podataka. U narednim notebook-ovima bice primenjeni KMeans, BIRCH, Agglomerative Clustering, Gaussian Mixture Model, Random Forest i XGBoost.